In [33]:
from base import generator_haar

def init_experiment(n):
    d = 2**n

    # Generate 6^n density matrices
    rho_list = generator_haar.generate_n_qubits_rho_haar(n)
    print(f"Generated {len(rho_list)} of {rho_list[0].shape} rho.")

    # Generate unitary
    unitary = generator_haar.random_unitary(d)
    print(f"Generated {unitary.shape} unitary operators.")
    return rho_list, unitary

In [47]:
import numpy as np
import tensorflow as tf

from base import epsilon_rho
def calculate_rho2_unitary(rho_list, unitary):
    rho2_unitary = []
    for rho in rho_list:
        rho2_unitary.append(epsilon_rho.calculate_from_unitary(rho, unitary))
    return rho2_unitary

def calculate_rho2_dephasing(rho_list, n, gamma):
    rho2 = []
    for rho in rho_list:
        rho2.append(epsilon_rho.calculate_dephasing(rho, n, gamma))
    return rho2

def write_to_file(filename, data):
    """Write TensorFlow tensor data to a text file without truncation."""
    tensor_data = data.numpy() if isinstance(data, tf.Tensor) else data

    # Open the file and write the tensor data
    with open(filename, 'w') as f:
        if isinstance(data, np.ndarray):
            np.savetxt(f, data, fmt="%.6f")
        elif isinstance(data, list):
            for item in data:
                f.write(f"{item}\n")
        else:
            f.write(str(data))


def normalize_unitary(matrix):
  
    
    # Perform QR decomposition to get the unitary matrix Q
    Q, _ = np.linalg.qr(matrix)
    
    return Q


In [54]:
from base import lost_func
def calculate_adam_unitary_dagger_set(rho_list, rho2_list, unitary, m, v, t, alpha=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8, mode = 'fidelity'):
    tensorUnitary = tf.Variable(unitary, dtype=tf.complex128)
    beta1 = tf.constant(beta1, dtype=tf.complex128)
    beta2 = tf.constant(beta2, dtype=tf.complex128)
    t = tf.constant(t, dtype=tf.complex128)

    with tf.GradientTape() as tape:
        data = epsilon_rho.calculate_set_from_unitary_dagger(tensorUnitary, rho2_list)
        f = lost_func.diff_MSE(data, rho_list)
    
    # Calculate the gradient
    c = tape.gradient(f, tensorUnitary)

    # Calculate projection
    proj = c - tensorUnitary @ (np.transpose(np.conjugate(c)) @ tensorUnitary + np.transpose(np.conjugate(tensorUnitary)) @ c) / 2

    # Update Adam variables
    m = beta1 * m + (1 - beta1) * proj
    v = beta2 * v + (1 - beta2) * tf.math.square(proj)

    # Bias correction
    m_hat = m / (1 - tf.pow(beta1, t + 1))
    v_hat = v / (1 - tf.pow(beta2, t + 1))

    # Update the Kraus operators using Adam update rule
    updated_unitary = tensorUnitary - alpha * m_hat / (tf.math.sqrt(v_hat) + epsilon)
    return updated_unitary, m, v, f

def optimize_adam_unitary_dagger_set(rho_list, rho2_list, unitary, num_qubits, alpha=0.001, beta1 = 0.9, beta2 = 0.999, epsilon = 1e-8, num_loop=1000):
    unitary_copy = tf.identity(unitary)
    
    # Initialize m, v to zero matrices
    m = tf.zeros_like(unitary_copy, dtype=tf.complex128)
    v = tf.zeros_like(unitary_copy, dtype=tf.complex128)
    
    # Initialize a dictionary to track cost at each iteration
    cost_dict = []

    # Try looping manually
    for i in range(num_loop):
        # Update Kraus Operators
        unitary_copy, m, v, cost = calculate_adam_unitary_dagger_set(rho_list=rho_list, rho2_list=rho2_list, unitary=unitary_copy, m = m, v = v, t = i, alpha=alpha, beta1=beta1, beta2=beta2, epsilon=epsilon)

        unitary_copy = normalize_unitary(unitary_copy)

        # Store the cost for this iteration
        cost_dict.append(cost.numpy().real)

    return unitary_copy, cost_dict

In [67]:
import os
from base import optimize_algorithm
from base import metrics
experiment_folder = 'results/experiment_new/dephasing'

g_shift = 0.05
for num_qubits in range(3, 4):
    if (experiment_folder == ''):
        break
    else:
        write_folder = os.path.join(experiment_folder, str(num_qubits) + "_qubits")
        if not os.path.exists(write_folder):
            os.makedirs(write_folder)
    print(f"N={num_qubits}")

    #-----Init experiment-----
    rho_list, unitary = init_experiment(num_qubits)
    write_to_file(os.path.join(write_folder, "rho_list.txt"), rho_list)

    g = 1
    while g >= 0:
        folder_path = os.path.join(write_folder, "_{:.2f}".format(g))
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)

        rho2_list = calculate_rho2_dephasing(rho_list, num_qubits, g)
    
        #-----Learn kraus operators-----
        unitary_res, cost_dict = optimize_adam_unitary_dagger_set(rho_list, rho2_list, unitary, num_qubits, 0.02, num_loop=200)
    
        #-----Calculate result data-----
        rho3_list = epsilon_rho.calculate_set_from_unitary_dagger(unitary_res, rho2_list)
        rho2_unitary_list = calculate_rho2_unitary(rho_list, unitary_res)
    
        mean_fidelity_rho_rho3 = metrics.mean_fidelity(rho3_list, rho_list)
        mean_fidelity_rho2_rho2 = metrics.mean_fidelity(rho2_unitary_list, rho2_list)

        #-----Write to folder-----    
        write_to_file(os.path.join(folder_path,"unitary.txt"), unitary)
        write_to_file(os.path.join(folder_path,"unitary_res.txt"), unitary_res)
        write_to_file(os.path.join(folder_path,"cost_dict.txt"), cost_dict)

        write_to_file(os.path.join(folder_path,"rho2_list.txt"), rho2_list)
        write_to_file(os.path.join(folder_path,"rho2_unitary_list.txt"), rho2_unitary_list)

        write_to_file(os.path.join(folder_path,"mean_fidelity_rho_rho3.txt"), mean_fidelity_rho_rho3.numpy())
        write_to_file(os.path.join(folder_path,"mean_fidelity_rho2_rho2.txt"), mean_fidelity_rho2_rho2.numpy())

        print(g, num_qubits)
        print(cost_dict[-1])
        g = round(g - g_shift, 2)

    
    

N=3
Generated 216 of (8, 8) rho.
Generated (8, 8) unitary operators.
1 3
0.7767712971943991
0.95 3
0.6404122070294652
0.9 3
0.5552926280189722
0.85 3
0.5013982963931136
0.8 3
0.4240165994982473
0.75 3
0.42589426901443933
0.7 3
0.3198085090562095
0.65 3
0.2786716668498733
0.6 3
0.23903556944460436
0.55 3
0.2592764271634457
0.5 3
0.18000101952661426
0.45 3
0.1729422945469265
0.4 3
0.17630266113501514
0.35 3
0.12405115116788545
0.3 3
0.11131083932891957
0.25 3
0.05436938574812529
0.2 3
0.09552069386797993
0.15 3
0.09286111872785219
0.1 3
0.11349724261069125
0.05 3
0.0783954281306178
0.0 3
0.05627808306867066
